# 04 — BiLSTM v8 (improved architecture and training)

What is new vs v7:

| Change | Why | Expected effect |
|---|---|---|
| `SEQ_LEN`: 7 → 4 | Most firms have short histories; longer windows lose positive samples. | Higher positive-sample coverage and potentially stronger PR-AUC. |
| 2-layer BiLSTM + LayerNorm + temporal attention | Deeper temporal representation with more stable training under focal loss. | Better temporal ranking ability. |
| Industry one-hot → `industry_idx` + embedding | Compact learnable representation of industry; `embed_dim` is selected by HPO. | Fewer sparse inputs and better regularisation. |
| Trend, volatility, minimum and maximum features | Three-year summaries for key ratios are added to the static branch. | Additional information about recent deterioration or instability. |
| 5-seed ensemble | Reduces single-seed variance. | More stable predictions. |
| SWA candidate | SWA predictions are compared with early-stopping predictions and only carried forward if validation PR-AUC is better. | Potential generalisation gain. |
| Isotonic calibration | Non-linear monotone calibration of the selected ensemble probabilities. | Better probability scale and Brier score. |
| Forced focal `α ∈ {0.70, 0.80, 0.85}` and `γ ∈ {1.0, 1.5, 2.0}` | Avoids degenerate low-positive-class weighting found in earlier HPO attempts. | More stable rare-event training. |

Pipeline:

1. Setup — Colab vs local, GPU detection.
2. Configuration — `SEQ_LEN=4`, out-of-time split, focal-loss grid and ensemble seeds.
3. Load `data_rnn.csv`.
4. Trend / volatility / min / max static features.
5. Industry index for the embedding layer.
6. Sequence + static + industry-index builder.
7. Custom focal loss.
8. Architecture: 2-layer BiLSTM + LayerNorm + temporal attention + industry embedding + static skip-through concatenation.
9. Balanced batch generator with three inputs.
10. SWA callback.
11. Optuna HPO with forced focal-α grid.
12. 5-seed ensemble training.
13. Evaluation — early-stopping ensemble vs SWA ensemble.
14. Isotonic calibration.
15. Threshold optimisation: Youden, F1 and F2.
16. Top-k lift, calibration metrics and bootstrap CIs for ROC-AUC / PR-AUC.
17. Per-year decomposition (2018 vs 2019).
18. Save predictions for downstream model comparison.
19. Single-seed retrain for SHAP analysis.


## 1. Setup — Colab vs local + GPU detection

Detects whether the notebook is running in Google Colab (mounts Drive if so), sets `BASE_DIR` from the environment variable `BASE_DIR` when available, configures GPU memory growth, and prints the active TensorFlow / Keras / NumPy versions.

In [ ]:
# Inštalácia tensorflow-addons bola odstránená, keďže knižnica nie je potrebná
# a spôsobuje konflikty verzií v novom TensorFlow.

In [ ]:
# === Detekcia prostredia ===
import os
try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = os.environ.get('BASE_DIR', '/content/drive/MyDrive/BILSTM')
    # Inštalácia chýbajúcich balíkov v Colab
    import subprocess, sys
    for pkg in ['optuna']:  # Odstránený tensorflow-addons
        try: __import__(pkg.replace('-','_'))
        except ImportError:
            subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
except ImportError:
    IS_COLAB = False
    BASE_DIR = os.environ.get('BASE_DIR', os.getcwd())

import os, time, pickle, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import backend as K, layers as L, Model
from tensorflow.keras.optimizers.schedules import CosineDecay

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, roc_curve, precision_recall_curve, fbeta_score
)
from scipy import stats as scistats
import matplotlib.pyplot as plt

print(f'Colab: {IS_COLAB} | TF: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
if gpus:
    try:
        for g in gpus: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

DATA_PATH = os.path.join(BASE_DIR, 'data_rnn.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'bilstm_v8')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'DATA: {DATA_PATH} (existuje: {os.path.exists(DATA_PATH)})')

## 2. Training configuration

Key settings: `SEED_LIST = [42, 7, 1337, 2025, 99]` for the 5-seed ensemble, `SEQ_LEN=4`, out-of-time split (train 2013–2016, valid 2017, test 2018–2019), focal-α candidates restricted to `{0.70, 0.80, 0.85}`, batch size, learning-rate schedule, early-stopping patience, SWA start epoch.

In [ ]:
SEED_LIST = [42, 7, 1337, 2025, 99]   # 5-seed ensemble
SEED = SEED_LIST[0]

# Out-of-time split
TRAIN_YEARS = [2013, 2014, 2015, 2016]
VALID_YEAR  = 2017
TEST_YEARS  = [2018, 2019]
OBS_YEARS   = TRAIN_YEARS + [VALID_YEAR] + TEST_YEARS

# === KĽÚČOVÉ ZMENY ===
SEQ_LEN = 4                              # bolo 7 — SHAP ukázalo T-7..T-4 ≈ 0
FOCAL_ALPHA_GRID = [0.70, 0.80, 0.85]    # bolo [0.50,...,0.90] — vylúčené 0.5
FOCAL_GAMMA_GRID = [1.0, 1.5, 2.0]
N_ENSEMBLE = 5                           # 5-seed ensemble

# Optuna HPO
N_TRIALS       = 30                      # menej, lebo užší grid
HPO_NEG_FRAC   = 0.25
HPO_MAX_EPOCHS = 25
HPO_PATIENCE   = 4

# Final fit
BATCH_SIZE   = 512
MAX_EPOCHS   = 80
FIN_PATIENCE = 12
SWA_START_EPOCH = 50                     # SWA od epochy 50

POS_BATCH_FRACTION = 0.25

np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Config OK.')

## 3. Load `data_rnn.csv`

Reads the sequence-ready dataset produced by notebook 00 (already filtered to `seq_len ≥ 2`, training-median imputed, with one-hot industry columns). Sorts by `(id, rok)` and reports shape and target rate.

In [ ]:
print('Načítavam ...')
data = pd.read_csv(DATA_PATH, low_memory=False)
data = data.sort_values(['id','rok']).reset_index(drop=True)
print(f'Shape: {data.shape} | target_rate: {data["target"].mean()*100:.3f}%')
print(f'Roky: {sorted(data["rok"].unique())}')

EXCLUDE_COLS = ['id','rok','target','seq_len']

_SEQ_LEVEL_CANDIDATES = [
    'cash_ta','eq_ta','ltd_ta','bl_ta','roa1','re_ta','roe','ros',
    'current_ratio','quick_ratio','debt_to_equity','debt_ratio','debt_ebitda',
    'ebitda_margin','interest_coverage','asset_turnover','creditors_days',
    'receivables_days','ebitda',
]
SEQ_LEVEL_FEATURES = [f for f in _SEQ_LEVEL_CANDIDATES if f in data.columns]
DYNAMIC_SUFFIXES = ('_lag1','_lag2','_diff1','_diff2','_yoy')
DYNAMIC_FEATURES = [c for c in data.columns
    if c not in EXCLUDE_COLS and c.endswith(DYNAMIC_SUFFIXES)]
SEQ_FEATURES = list(dict.fromkeys(SEQ_LEVEL_FEATURES + DYNAMIC_FEATURES))

# Statické: všetko ostatné okrem sekvenčných
TEXT_HELPER = {'industry_group','industry_slug','nace_section','sk_nace','sknace',
               'pravna_forma','datum_vzniku'}
ALL_NUM = [c for c in data.columns
    if c not in EXCLUDE_COLS and c not in TEXT_HELPER and c not in SEQ_FEATURES
    and pd.api.types.is_numeric_dtype(data[c])]

# Identifikuj industry dummy stĺpce
INDUSTRY_COLS = sorted([c for c in ALL_NUM if c.startswith('industry_')])
# Ostatné statické (bez industry)
OTHER_STATIC = [c for c in ALL_NUM if not c.startswith('industry_')]

print(f'Seq features: {len(SEQ_FEATURES)} (level {len(SEQ_LEVEL_FEATURES)} + dynamic {len(DYNAMIC_FEATURES)})')
print(f'Industry dummies: {len(INDUSTRY_COLS)}')
print(f'Other static: {len(OTHER_STATIC)}')

## 4. Trend & volatility static features (new in v8)

For each key ratio (`cash_ta`, `eq_ta`, `roa1`, `current_ratio`, `debt_ratio`) and for each firm-year, computes a 3-year linear-regression slope (`slope3y`), year-over-year volatility (`vol3y`), rolling minimum (`min3y`) and rolling maximum (`max3y`) using the up-to-current-year window. These additional summaries enter the **static** branch of the network.

In [ ]:
KEY_RATIOS_FOR_TRENDS = ['cash_ta','eq_ta','roa1','current_ratio','debt_ratio']
KEY_RATIOS_FOR_TRENDS = [c for c in KEY_RATIOS_FOR_TRENDS if c in data.columns]

def add_trend_features(df, ratios, window=3):
    df = df.sort_values(['id','rok']).copy()
    new_cols = []
    for r in ratios:
        # rolling slope: pre každý riadok zoberie posledných `window` hodnôt
        # a fitne lineárny trend
        slopes, vols, mins, maxs = [], [], [], []
        for fid, grp in df.groupby('id'):
            g = grp[r].values
            n = len(g)
            sl = np.full(n, np.nan); vo = np.full(n, np.nan)
            mn = np.full(n, np.nan); mx = np.full(n, np.nan)
            for i in range(n):
                lo = max(0, i-window+1)
                w = g[lo:i+1]
                w = w[~np.isnan(w)]
                if len(w) >= 2:
                    x = np.arange(len(w))
                    sl[i] = float(np.polyfit(x, w, 1)[0])
                    yoy = np.diff(w) / (np.abs(w[:-1])+1e-6)
                    vo[i] = float(np.std(yoy)) if len(yoy)>1 else 0.0
                    mn[i] = float(np.min(w)); mx[i] = float(np.max(w))
            slopes.append(pd.Series(sl, index=grp.index))
            vols.append(pd.Series(vo, index=grp.index))
            mins.append(pd.Series(mn, index=grp.index))
            maxs.append(pd.Series(mx, index=grp.index))
        df[f'{r}_slope3y'] = pd.concat(slopes).sort_index()
        df[f'{r}_vol3y']   = pd.concat(vols).sort_index()
        df[f'{r}_min3y']   = pd.concat(mins).sort_index()
        df[f'{r}_max3y']   = pd.concat(maxs).sort_index()
        new_cols += [f'{r}_slope3y', f'{r}_vol3y', f'{r}_min3y', f'{r}_max3y']
    return df, new_cols

print('Počítam trend/volatility features ...')
t0 = time.time()
data, TREND_COLS = add_trend_features(data, KEY_RATIOS_FOR_TRENDS, window=3)
print(f'  hotovo za {time.time()-t0:.1f}s; pridaných {len(TREND_COLS)} stĺpcov')

# Imputácia NaN (mediánom z train rokov)
train_mask = data['rok'].isin(TRAIN_YEARS)
for c in TREND_COLS:
    med = data.loc[train_mask, c].median()
    data[c] = data[c].fillna(med)

# Pridaj trend cols do statickej vetvy
OTHER_STATIC = OTHER_STATIC + TREND_COLS
print(f'OTHER_STATIC po doplnení: {len(OTHER_STATIC)}')

## 5. Industry index for the embedding layer (new in v8)

Replaces the industry one-hot block with a single integer `industry_idx`. Firms are assigned to the active industry dummy; observations without an active dummy receive a dedicated missing/unknown index. The neural network then uses an embedding layer with `input_dim = N_INDUSTRY + 1` and `embed_dim` selected by Optuna.


In [ ]:
# Vyrobíme industry index z one-hot stĺpcov
def industry_index(row, industry_cols):
    for i, c in enumerate(industry_cols):
        if row[c] == 1:
            return i
    return len(industry_cols)  # fallback = 'unknown'

print('Vytváram industry_idx ...')
data['industry_idx'] = data[INDUSTRY_COLS].values.argmax(axis=1).astype(np.int32)
# kontrola - ak žiadny stĺpec nemá 1, idx = 0 (treba ošetriť)
no_match = (data[INDUSTRY_COLS].sum(axis=1) == 0)
data.loc[no_match, 'industry_idx'] = len(INDUSTRY_COLS)  # extra index = 'missing'
N_INDUSTRY = data['industry_idx'].max() + 1
print(f'N_INDUSTRY: {N_INDUSTRY}')

# Statické features už NEZAHŔŇAJÚ industry one-hot
STAT_FEATURES_NO_INDUSTRY = OTHER_STATIC  # už bez industry_*
print(f'STAT_FEATURES (no industry): {len(STAT_FEATURES_NO_INDUSTRY)}')

## 6. Sequence + static + industry-index builder

`build_sequences_v8` walks each firm history, slides a fixed-length window of length `SEQ_LEN` over the sequence-level features, and emits four aligned arrays per observation year: `X_seq` (B, SEQ_LEN, n_seq), `X_ind` (B,) industry index, `X_stat` (B, n_static) static features and `y` (B,) target. Pre-window history is left-padded with the masking value so the `Masking` layer can ignore it.

In [ ]:
def build_sequences_v8(df, obs_years, seq_feats, stat_feats, seq_len):
    firm_hist = {fid: grp.sort_values('rok').reset_index(drop=True)
                 for fid, grp in df.groupby('id', sort=False)}
    obs_df = df[df['rok'].isin(obs_years)].copy()

    X_seq_raw, X_stat, ind_idx, y_out, meta = [], [], [], [], []
    for _, row in obs_df.iterrows():
        fid, rok_t = row['id'], row['rok']
        if fid not in firm_hist: continue
        hist = firm_hist[fid]
        hist = hist[hist['rok'] <= rok_t].tail(seq_len)
        if len(hist) < 1: continue
        X_seq_raw.append(hist[seq_feats].values.astype(np.float32))
        X_stat.append(hist.iloc[-1][stat_feats].values.astype(np.float32))
        ind_idx.append(int(hist.iloc[-1]['industry_idx']))
        y_out.append(int(row['target']))
        meta.append((int(fid), int(rok_t)))
    return (X_seq_raw,
            np.array(X_stat, dtype=np.float32),
            np.array(ind_idx, dtype=np.int32),
            np.array(y_out, dtype=np.float32),
            meta)

def scale_and_pad(seq_list, scaler, seq_len):
    n = len(seq_list); n_feat = seq_list[0].shape[1] if seq_list else 1
    out = np.zeros((n, seq_len, n_feat), dtype=np.float32)
    for i, s in enumerate(seq_list):
        sc = scaler.transform(s); t = sc.shape[0]
        out[i, seq_len-t:, :] = sc
    return out

print('Build sequences ...')
t0 = time.time()
X_seq_raw, X_stat_raw, ind_all, y_all, meta_all = build_sequences_v8(
    data, OBS_YEARS, SEQ_FEATURES, STAT_FEATURES_NO_INDUSTRY, SEQ_LEN)
print(f'Build: {time.time()-t0:.1f}s | n={len(y_all):,} | bk={int(y_all.sum())} ({y_all.mean()*100:.3f}%)')

meta_arr = np.array(meta_all)
rok_col = meta_arr[:,1]

mask_tr = np.isin(rok_col, TRAIN_YEARS)
mask_va = (rok_col == VALID_YEAR)
mask_te = np.isin(rok_col, TEST_YEARS)

# Sekvencie
X_seq_tr_raw = [X_seq_raw[i] for i in np.where(mask_tr)[0]]
X_seq_va_raw = [X_seq_raw[i] for i in np.where(mask_va)[0]]
X_seq_te_raw = [X_seq_raw[i] for i in np.where(mask_te)[0]]
# Statické
X_st_tr_raw = X_stat_raw[mask_tr]
X_st_va_raw = X_stat_raw[mask_va]
X_st_te_raw = X_stat_raw[mask_te]
# Industry idx
ind_tr = ind_all[mask_tr]
ind_va = ind_all[mask_va]
ind_te = ind_all[mask_te]
# Targets
y_train = y_all[mask_tr]; y_valid = y_all[mask_va]; y_test = y_all[mask_te]
# Meta
id_valid = meta_arr[mask_va, 0]; id_test = meta_arr[mask_te, 0]
rok_valid = meta_arr[mask_va, 1]; rok_test = meta_arr[mask_te, 1]

# Scaling — fit ONLY on train
scaler_seq  = StandardScaler().fit(np.vstack(X_seq_tr_raw))
scaler_stat = StandardScaler().fit(X_st_tr_raw)

X_seq_train = scale_and_pad(X_seq_tr_raw, scaler_seq, SEQ_LEN)
X_seq_valid = scale_and_pad(X_seq_va_raw, scaler_seq, SEQ_LEN)
X_seq_test  = scale_and_pad(X_seq_te_raw, scaler_seq, SEQ_LEN)
X_st_train  = scaler_stat.transform(X_st_tr_raw).astype(np.float32)
X_st_valid  = scaler_stat.transform(X_st_va_raw).astype(np.float32)
X_st_test   = scaler_stat.transform(X_st_te_raw).astype(np.float32)

print(f'TRAIN seq:{X_seq_train.shape} stat:{X_st_train.shape} ind:{ind_tr.shape} y:{y_train.shape}')
print(f'VALID seq:{X_seq_valid.shape} stat:{X_st_valid.shape} ind:{ind_va.shape}')
print(f'TEST  seq:{X_seq_test.shape}  stat:{X_st_test.shape}  ind:{ind_te.shape}')

## 7. Focal loss (custom implementation, no label smoothing)

`FocalLoss` re-weights binary cross-entropy by `(1−p_t)^γ` so the gradient focuses on hard examples. The class-weighting parameter `α` and focusing parameter `γ` have defaults in the class definition but the final values are selected by Optuna from the configured grids. Label smoothing is deliberately omitted because it would complicate the calibration step downstream.


In [ ]:
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.80, name='focal_loss', **kw):
        super().__init__(name=name, **kw)
        self.gamma, self.alpha = float(gamma), float(alpha)
    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true,[-1]), tf.float32)
        y_pred = tf.cast(tf.reshape(y_pred,[-1]), tf.float32)
        eps = K.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1-eps)
        p_t = tf.where(y_true==1.0, y_pred, 1.0 - y_pred)
        alpha_t = tf.where(y_true==1.0,
            tf.fill(tf.shape(y_true), self.alpha),
            tf.fill(tf.shape(y_true), 1.0-self.alpha))
        focal_w = tf.pow(1.0 - p_t, self.gamma)
        return tf.reduce_mean(-alpha_t * focal_w * tf.math.log(p_t))

## 8. Architecture — 2-layer BiLSTM + LayerNorm + industry embedding + static skip-through (new in v8)

```
seq_in (B, 4, n_seq)
  -> Masking -> GaussianNoise -> LayerNorm
  -> BiLSTM(lstm1, return_sequences=True)
  -> LayerNorm
  -> BiLSTM(lstm2, return_sequences=True)
  -> TemporalAttention -> LayerNorm -> Dropout

ind_in (B,)
  -> Embedding(input_dim=N_INDUSTRY+1, output_dim=embed_dim)
  -> Flatten

stat_in (B, n_stat)
  -> Dense(24, relu) -> BatchNorm -> Dropout

concat(seq_branch, industry_embedding, static_branch, raw_scaled_stat_in)
  -> Dense(dense_units, relu) -> BatchNorm -> Dropout
  -> Dense(1, sigmoid)
```

`lstm1`, `lstm2`, `embed_dim`, `dropout`, `dense_units`, learning rate and focal-loss parameters are selected by Optuna. `TemporalAttention` produces one attention score per time step before pooling.


In [ ]:
class TemporalAttention(tf.keras.layers.Layer):
    def __init__(self, **kw):
        super().__init__(**kw)
        self._score = L.Dense(1, use_bias=True, name='attn_score')
    def call(self, x, mask=None):
        scores = tf.cast(self._score(x), tf.float32)
        if mask is not None:
            mask_exp = tf.cast(tf.expand_dims(mask,-1), tf.float32)
            scores = scores + (1.0 - mask_exp) * (-1e9)
        w = tf.nn.softmax(scores, axis=1)
        w = tf.cast(w, x.dtype)
        return tf.reduce_sum(x * w, axis=1)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])
    def get_config(self): return super().get_config()


def build_bilstm_v8(seq_len, n_seq, n_stat, n_industry,
                    embed_dim=8, lstm1=32, lstm2=32,
                    dropout_rate=0.30, dense_units=64,
                    learning_rate=2e-3, focal_gamma=2.0, focal_alpha=0.80,
                    max_epochs=80):
    # === Sekvenčná vetva: 2-layer BiLSTM ===
    seq_in = L.Input(shape=(seq_len, n_seq), name='seq_input')
    x = L.Masking(mask_value=0.0)(seq_in)
    x = L.GaussianNoise(0.01)(x)
    x = L.LayerNormalization(name='ln_in')(x)
    x = L.Bidirectional(
            L.LSTM(lstm1, return_sequences=True, use_cudnn=False,
                   dropout=dropout_rate, recurrent_dropout=0.0),
            name='bilstm_1')(x)
    x = L.LayerNormalization(name='ln_mid')(x)
    x = L.Bidirectional(
            L.LSTM(lstm2, return_sequences=True, use_cudnn=False,
                   dropout=dropout_rate, recurrent_dropout=0.0),
            name='bilstm_2')(x)
    x = TemporalAttention(name='temporal_attn')(x)
    x = L.LayerNormalization(name='ln_attn')(x)
    x = L.Dropout(dropout_rate, name='drop_seq')(x)

    # === Industry embedding ===
    ind_in = L.Input(shape=(1,), dtype='int32', name='ind_input')
    ind_emb = L.Embedding(n_industry+1, embed_dim, name='ind_embedding')(ind_in)
    ind_emb = L.Flatten(name='ind_flatten')(ind_emb)

    # === Statická vetva (bez industry) ===
    stat_in = L.Input(shape=(n_stat,), name='stat_input')
    s = L.Dense(24, activation='relu', name='stat_dense')(stat_in)
    s = L.BatchNormalization(name='bn_stat')(s)
    s = L.Dropout(dropout_rate/2, name='drop_stat')(s)

    # === SKIP connection: stat_in priamo do hlavy ===
    merged = L.Concatenate(name='concat')([x, ind_emb, s, stat_in])
    h = L.Dense(dense_units, activation='relu', name='head_dense')(merged)
    h = L.BatchNormalization(name='bn_head')(h)
    h = L.Dropout(dropout_rate, name='drop_head')(h)
    out = L.Dense(1, activation='sigmoid', dtype='float32', name='output')(h)

    model = Model(inputs=[seq_in, ind_in, stat_in], outputs=out, name='bilstm_v8')

    # Cosine LR schedule
    steps_per_epoch_est = max(1, len(y_train) // BATCH_SIZE)
    decay_steps = max_epochs * steps_per_epoch_est
    lr_schedule = CosineDecay(
        initial_learning_rate=learning_rate,
        decay_steps=decay_steps,
        alpha=0.05  # finálna LR = 5% pôvodnej
    )
    opt = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0)
    model.compile(
        optimizer=opt,
        loss=FocalLoss(gamma=focal_gamma, alpha=focal_alpha),
        metrics=[tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.AUC(name='auc_pr', curve='PR')]
    )
    return model

print('Architektúra v8 pripravená.')

## 9. Balanced batch generator with three inputs

`make_balanced_dataset_v8` yields `tf.data.Dataset` batches in which the positive-class share is fixed at `pos_fraction` (default 0.25). It samples positive and negative indices independently and assembles `((X_seq, X_ind, X_stat), y)` tuples with the right shapes for the multi-input model.

In [ ]:
def make_balanced_dataset_v8(X_seq, ind, X_stat, y, batch_size=512,
                              pos_fraction=0.25, seed=42):
    y = np.asarray(y).astype(np.float32)
    pos_idx = np.where(y==1)[0]; neg_idx = np.where(y==0)[0]
    pos_bs = max(1, int(round(batch_size * pos_fraction)))
    neg_bs = max(1, batch_size - pos_bs)

    ds_pos = tf.data.Dataset.from_tensor_slices(
        ((X_seq[pos_idx], ind[pos_idx], X_stat[pos_idx]), y[pos_idx])
    ).shuffle(len(pos_idx), seed=seed, reshuffle_each_iteration=True).repeat().batch(pos_bs)

    ds_neg = tf.data.Dataset.from_tensor_slices(
        ((X_seq[neg_idx], ind[neg_idx], X_stat[neg_idx]), y[neg_idx])
    ).shuffle(min(len(neg_idx), 100_000), seed=seed+1,
              reshuffle_each_iteration=True).repeat().batch(neg_bs)

    def merge(p, n):
        (xs_p, xi_p, xt_p), y_p = p; (xs_n, xi_n, xt_n), y_n = n
        return ((tf.concat([xs_p, xs_n], 0),
                 tf.concat([xi_p, xi_n], 0),
                 tf.concat([xt_p, xt_n], 0)),
                tf.concat([y_p, y_n], 0))

    ds = tf.data.Dataset.zip((ds_pos, ds_neg)).map(merge).prefetch(tf.data.AUTOTUNE)
    steps = max(1, len(y) // batch_size)
    return ds, steps

print('Balanced generator pripravený.')

## 10. Stochastic Weight Averaging callback (new in v8)

`SWACallback` averages the model weights from `start_epoch` until the end of training. SWA can find wider, flatter minima; the notebook therefore compares SWA against the early-stopping ensemble and carries forward the better option by validation PR-AUC.

In [ ]:
class SWACallback(tf.keras.callbacks.Callback):
    """Priemeruje weights od epochy `start_epoch` do konca tréningu."""
    def __init__(self, start_epoch=50):
        super().__init__()
        self.start_epoch = start_epoch
        self.weights_sum = None
        self.n_avg = 0
    def on_epoch_end(self, epoch, logs=None):
        if epoch < self.start_epoch: return
        cur = self.model.get_weights()
        if self.weights_sum is None:
            self.weights_sum = [w.copy() for w in cur]
            self.n_avg = 1
        else:
            for i, w in enumerate(cur):
                self.weights_sum[i] += w
            self.n_avg += 1
    def get_swa_weights(self):
        if self.weights_sum is None or self.n_avg == 0:
            return None
        return [w / self.n_avg for w in self.weights_sum]

print('SWA callback pripravený.')

## 11. Optuna HPO with forced focal-α grid

Runs a small Optuna search (TPE) on a sub-sample (`HPO_NEG_FRAC` of the negative class). The focal-α grid is restricted to `{0.70, 0.80, 0.85}` to avoid the degenerate α < 0.5 region observed in v7 HPO. Validation PR-AUC is the objective.

In [ ]:
# Sub-vzorka pre HPO
pos_idx = np.where(y_train==1)[0]; neg_idx = np.where(y_train==0)[0]
n_neg = int(len(neg_idx) * HPO_NEG_FRAC)
rng = np.random.default_rng(SEED)
neg_sub = rng.choice(neg_idx, n_neg, replace=False)
sub_idx = np.concatenate([pos_idx, neg_sub]); rng.shuffle(sub_idx)

X_seq_sub = X_seq_train[sub_idx]; ind_sub = ind_tr[sub_idx]
X_st_sub  = X_st_train[sub_idx];  y_sub   = y_train[sub_idx]

hpo_ds, hpo_steps = make_balanced_dataset_v8(
    X_seq_sub, ind_sub, X_st_sub, y_sub,
    batch_size=BATCH_SIZE, pos_fraction=POS_BATCH_FRACTION, seed=SEED+100)

def objective(trial):
    lstm1 = trial.suggest_categorical('lstm1', [16, 32, 48])
    lstm2 = trial.suggest_categorical('lstm2', [16, 32])
    dropout = trial.suggest_float('dropout', 0.20, 0.40)
    dense = trial.suggest_categorical('dense', [32, 64, 96])
    lr = trial.suggest_float('lr', 5e-4, 5e-3, log=True)
    embed_dim = trial.suggest_categorical('embed_dim', [4, 8, 12])
    fg = trial.suggest_categorical('focal_gamma', FOCAL_GAMMA_GRID)
    fa = trial.suggest_categorical('focal_alpha', FOCAL_ALPHA_GRID)  # vynútené ≥0.70

    K.clear_session()
    m = build_bilstm_v8(SEQ_LEN, len(SEQ_FEATURES), len(STAT_FEATURES_NO_INDUSTRY),
                        N_INDUSTRY, embed_dim=embed_dim, lstm1=lstm1, lstm2=lstm2,
                        dropout_rate=dropout, dense_units=dense,
                        learning_rate=lr, focal_gamma=fg, focal_alpha=fa,
                        max_epochs=HPO_MAX_EPOCHS)
    cb = [tf.keras.callbacks.EarlyStopping(monitor='val_auc_pr', patience=HPO_PATIENCE,
                                            mode='max', restore_best_weights=True, verbose=0)]
    m.fit(hpo_ds, validation_data=([X_seq_valid, ind_va, X_st_valid], y_valid),
          epochs=HPO_MAX_EPOCHS, steps_per_epoch=hpo_steps,
          callbacks=cb, verbose=0)
    pv = m.predict([X_seq_valid, ind_va, X_st_valid], batch_size=2048, verbose=0).ravel()
    pr = average_precision_score(y_valid, pv)
    K.clear_session(); del m
    return pr

study = optuna.create_study(direction='maximize',
    sampler=TPESampler(seed=SEED, n_startup_trials=5),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    study_name='bilstm_v8')
print(f'HPO {N_TRIALS} trialov ...')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
best = study.best_params; best_val = study.best_value
print(f'\nBest val PR-AUC: {best_val:.5f}')
for k,v in best.items(): print(f'  {k}: {v}')

## 12. 5-seed ensemble training (new in v8)

Trains the same architecture five times with different seeds on the full data and balanced batches. Each run produces both an early-stopping checkpoint and an SWA-averaged checkpoint, and stores their validation / test predictions for later averaging.

In [ ]:
ensemble_probs_valid = []
ensemble_probs_test  = []
ensemble_swa_probs_valid = []
ensemble_swa_probs_test  = []
ensemble_metrics = []

for seed_i, seed in enumerate(SEED_LIST):
    print(f'\n{"="*60}\n SEED {seed_i+1}/{len(SEED_LIST)}: {seed}\n{"="*60}')
    np.random.seed(seed); tf.random.set_seed(seed)
    K.clear_session()

    train_ds, train_steps = make_balanced_dataset_v8(
        X_seq_train, ind_tr, X_st_train, y_train,
        batch_size=BATCH_SIZE, pos_fraction=POS_BATCH_FRACTION, seed=seed)

    model = build_bilstm_v8(
        SEQ_LEN, len(SEQ_FEATURES), len(STAT_FEATURES_NO_INDUSTRY), N_INDUSTRY,
        embed_dim=best['embed_dim'], lstm1=best['lstm1'], lstm2=best['lstm2'],
        dropout_rate=best['dropout'], dense_units=best['dense'],
        learning_rate=best['lr'],
        focal_gamma=best['focal_gamma'], focal_alpha=best['focal_alpha'],
        max_epochs=MAX_EPOCHS)

    es = tf.keras.callbacks.EarlyStopping(monitor='val_auc_pr', patience=FIN_PATIENCE,
        mode='max', restore_best_weights=True, verbose=1)
    swa = SWACallback(start_epoch=SWA_START_EPOCH)

    hist = model.fit(train_ds, validation_data=([X_seq_valid, ind_va, X_st_valid], y_valid),
        epochs=MAX_EPOCHS, steps_per_epoch=train_steps,
        callbacks=[es, swa], verbose=2)

    # Štandardná predikcia
    pv = model.predict([X_seq_valid, ind_va, X_st_valid], batch_size=2048, verbose=0).ravel()
    pt = model.predict([X_seq_test,  ind_te, X_st_test],  batch_size=2048, verbose=0).ravel()
    ensemble_probs_valid.append(pv); ensemble_probs_test.append(pt)

    auc_v = roc_auc_score(y_valid, pv); pr_v = average_precision_score(y_valid, pv)
    auc_t = roc_auc_score(y_test, pt);  pr_t = average_precision_score(y_test, pt)
    print(f'  [seed {seed} ES] valid AUC={auc_v:.4f} PR={pr_v:.4f} | test AUC={auc_t:.4f} PR={pr_t:.4f}')

    # SWA predikcia
    swa_w = swa.get_swa_weights()
    if swa_w is not None:
        model.set_weights(swa_w)
        pv_swa = model.predict([X_seq_valid, ind_va, X_st_valid], batch_size=2048, verbose=0).ravel()
        pt_swa = model.predict([X_seq_test,  ind_te, X_st_test],  batch_size=2048, verbose=0).ravel()
        ensemble_swa_probs_valid.append(pv_swa); ensemble_swa_probs_test.append(pt_swa)
        auc_sv = roc_auc_score(y_valid, pv_swa); pr_sv = average_precision_score(y_valid, pv_swa)
        auc_st = roc_auc_score(y_test, pt_swa);  pr_st = average_precision_score(y_test, pt_swa)
        print(f'  [seed {seed} SWA] valid AUC={auc_sv:.4f} PR={pr_sv:.4f} | test AUC={auc_st:.4f} PR={pr_st:.4f}')

    ensemble_metrics.append({'seed':seed, 'es_auc_test':auc_t, 'es_pr_test':pr_t})
    del model; K.clear_session()

# Priemerovanie
probs_valid_ens = np.mean(np.stack(ensemble_probs_valid), axis=0)
probs_test_ens  = np.mean(np.stack(ensemble_probs_test),  axis=0)
if ensemble_swa_probs_valid:
    probs_valid_swa_ens = np.mean(np.stack(ensemble_swa_probs_valid), axis=0)
    probs_test_swa_ens  = np.mean(np.stack(ensemble_swa_probs_test),  axis=0)
else:
    probs_valid_swa_ens = probs_valid_ens.copy()
    probs_test_swa_ens  = probs_test_ens.copy()

## 13. Ensemble evaluation — ES vs SWA

`eval_block` reports ROC-AUC, PR-AUC and KS for each individual seed and for the ensemble (mean of probabilities), comparing the early-stopping checkpoint against the SWA-averaged checkpoint. The better of the two — by validation PR-AUC — is then carried forward.

In [ ]:
def eval_block(name, y, p, base=None):
    fpr, tpr, _ = roc_curve(y, p)
    out = {
        'name': name,
        'AUC': roc_auc_score(y, p),
        'PR_AUC': average_precision_score(y, p),
        'KS': float(np.max(tpr - fpr)),
        'Brier': brier_score_loss(y, p),
    }
    if base is not None:
        out['ΔAUC'] = out['AUC'] - base['AUC']
        out['ΔPR'] = out['PR_AUC'] - base['PR_AUC']
    return out

print('=== Per-seed test ===')
for m in ensemble_metrics:
    print(f'  seed {m["seed"]:>5}: AUC={m["es_auc_test"]:.4f}  PR={m["es_pr_test"]:.5f}')

rows = [eval_block('Single (seed 42)', y_test, ensemble_probs_test[0]),
        eval_block('5-seed ES ensemble', y_test, probs_test_ens)]
if ensemble_swa_probs_valid:
    rows.append(eval_block('5-seed SWA ensemble', y_test, probs_test_swa_ens))

eval_df = pd.DataFrame(rows).round(5)
print('\n=== ENSEMBLE TEST ===')
print(eval_df.to_string(index=False))

## 14. Isotonic calibration (new in v8)

Replaces Platt scaling with isotonic regression on the validation predictions. Isotonic is monotone but non-linear, so it can correct miscalibration that Platt cannot.

In [ ]:
# Vyber lepší zo {ES, SWA} podľa val PR-AUC
pr_es  = average_precision_score(y_valid, probs_valid_ens)
pr_swa = average_precision_score(y_valid, probs_valid_swa_ens)
if pr_swa > pr_es:
    print(f'SWA zvíťazil: val PR-AUC {pr_swa:.4f} > ES {pr_es:.4f}')
    probs_valid_best = probs_valid_swa_ens; probs_test_best = probs_test_swa_ens
    chosen = 'SWA'
else:
    print(f'ES zvíťazil: val PR-AUC {pr_es:.4f} >= SWA {pr_swa:.4f}')
    probs_valid_best = probs_valid_ens; probs_test_best = probs_test_ens
    chosen = 'ES'

iso = IsotonicRegression(out_of_bounds='clip').fit(probs_valid_best, y_valid)
probs_valid_cal = iso.transform(probs_valid_best)
probs_test_cal  = iso.transform(probs_test_best)

print('\n=== KALIBRÁCIA ===')
print(f'  Brier raw : {brier_score_loss(y_test, probs_test_best):.6f}')
print(f'  Brier iso : {brier_score_loss(y_test, probs_test_cal):.6f}')
print(f'  AUC raw   : {roc_auc_score(y_test, probs_test_best):.4f}')
print(f'  AUC iso   : {roc_auc_score(y_test, probs_test_cal):.4f}  (mal by byť rovnaký)')

## 15. Threshold optimisation: Youden, F1 and F2

Given the calibrated validation predictions, the notebook selects three candidate thresholds: Youden-J, max-F1 and max-F2 (recall-weighted). All three thresholds are evaluated on the test set.


In [ ]:
def f_beta(y, p, thr, beta=2):
    pred = (p >= thr).astype(int)
    return fbeta_score(y, pred, beta=beta, zero_division=0)

# Z validácie
fpr, tpr, thrs = roc_curve(y_valid, probs_valid_cal)
youden_thr = float(thrs[np.argmax(tpr - fpr)])

# F2 (recall-favoring)
f2_grid = [(t, f_beta(y_valid, probs_valid_cal, t, beta=2)) for t in np.linspace(0.001, 0.5, 200)]
f2_thr = max(f2_grid, key=lambda r: r[1])[0]

# F1
f1_grid = [(t, f_beta(y_valid, probs_valid_cal, t, beta=1)) for t in np.linspace(0.001, 0.5, 200)]
f1_thr = max(f1_grid, key=lambda r: r[1])[0]

print('Prahy (validácia):')
print(f'  Youden: {youden_thr:.4f}')
print(f'  F1-opt: {f1_thr:.4f}')
print(f'  F2-opt: {f2_thr:.4f}')

# Aplikuj na test
def confusion_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {'thr':thr, 'TP':int(tp),'FP':int(fp),'FN':int(fn),'TN':int(tn),
            'recall':tp/(tp+fn) if tp+fn>0 else 0,
            'precision':tp/(tp+fp) if tp+fp>0 else 0,
            'F1':f_beta(y,p,thr,1), 'F2':f_beta(y,p,thr,2)}

thresh_table = pd.DataFrame([
    {'name':'Youden', **confusion_metrics(y_test, probs_test_cal, youden_thr)},
    {'name':'F1-opt', **confusion_metrics(y_test, probs_test_cal, f1_thr)},
    {'name':'F2-opt', **confusion_metrics(y_test, probs_test_cal, f2_thr)},
])
print('\n=== Test pri rôznych prahoch ===')
print(thresh_table.round(4).to_string(index=False))

## 16. Top-k lift, calibration metrics and bootstrap CIs

Reports lift at top 1 / 5 / 10 %, calibration slope and intercept on the test set, Brier score, and stratified bootstrap confidence intervals for ROC-AUC and PR-AUC.


In [ ]:
def topk_lift(y, p, ks=(0.01, 0.05, 0.10)):
    n = len(y); pos = y.sum()
    order = np.argsort(p)[::-1]
    out = []
    for k in ks:
        cut = max(int(np.ceil(n*k)), 1)
        cap = int(y[order[:cut]].sum())
        out.append({'k':k, 'cut':cut, 'captured':cap,
                    'capture_rate':cap/pos if pos>0 else 0,
                    'lift':(cap/cut)/(pos/n) if pos>0 else 0})
    return pd.DataFrame(out)

print('=== TOP-K LIFT ===')
print(topk_lift(y_test, probs_test_cal).round(3).to_string(index=False))

# Calibration slope/intercept
def cal_slope(y, p, eps=1e-9):
    p = np.clip(p, eps, 1-eps); lp = np.log(p/(1-p))
    import statsmodels.api as sm
    X = sm.add_constant(lp)
    try:
        r = sm.Logit(y, X).fit(disp=False, maxiter=500)
        return {'intercept':float(r.params[0]), 'slope':float(r.params[1])}
    except: return {'intercept':np.nan, 'slope':np.nan}

cs_raw = cal_slope(y_test, probs_test_best)
cs_iso = cal_slope(y_test, probs_test_cal)
print(f'\n=== Calibration (test) ===')
print(f'  Raw : intercept={cs_raw["intercept"]:.3f} slope={cs_raw["slope"]:.3f}')
print(f'  Iso : intercept={cs_iso["intercept"]:.3f} slope={cs_iso["slope"]:.3f} (ideál 0,1)')

# Bootstrap CI
def bootstrap_ci(y, p, n=500, seed=42):
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y==1)[0]; neg_idx = np.where(y==0)[0]
    aucs, prs = [], []
    for _ in range(n):
        sp = rng.choice(pos_idx, len(pos_idx), replace=True)
        sn = rng.choice(neg_idx, len(neg_idx), replace=True)
        idx = np.concatenate([sp, sn])
        try:
            aucs.append(roc_auc_score(y[idx], p[idx]))
            prs.append(average_precision_score(y[idx], p[idx]))
        except: continue
    return {
        'AUC':(np.mean(aucs), np.percentile(aucs,2.5), np.percentile(aucs,97.5)),
        'PR_AUC':(np.mean(prs), np.percentile(prs,2.5), np.percentile(prs,97.5)),
    }

print('\nBootstrap 500x stratifikovaný ...')
bci = bootstrap_ci(y_test, probs_test_cal, n=500)
for k,(mu,lo,hi) in bci.items():
    print(f'  {k}: {mu:.4f} [{lo:.4f}, {hi:.4f}]')

## 17. Per-year decomposition (2018 vs 2019)

Separate ROC-AUC, PR-AUC and KS for 2018 and 2019, with a short read on temporal stability of the model.

In [ ]:
rows = []
for yr in [2018, 2019]:
    m = rok_test == yr
    if m.sum() == 0: continue
    yt = y_test[m]; pt = probs_test_cal[m]
    fpr, tpr, _ = roc_curve(yt, pt)
    rows.append({
        'rok': int(yr), 'n': int(m.sum()), 'bk': int(yt.sum()),
        'AUC': roc_auc_score(yt, pt),
        'PR_AUC': average_precision_score(yt, pt),
        'KS': float(np.max(tpr-fpr)),
        'Brier': brier_score_loss(yt, pt),
    })
per_year_df = pd.DataFrame(rows).round(4)
print(per_year_df.to_string(index=False))

## 18. Save predictions and metadata for the stacking notebook

Writes the calibrated validation and test probabilities, ids, target, chosen ensemble (ES vs SWA), best HPO hyperparameters and the per-threshold metrics to `bilstm_v8_results.pkl`. A backward-compatible alias `bilstm_v7_results.pkl` is also written for older downstream comparison notebooks.

In [ ]:
out = {
    'model_name': 'BiLSTM v8 — 2-layer + LayerNorm + IndustryEmbed + Trend + 5-seed + SWA + Iso',
    'version': 'v8',
    'best_hparams': dict(best),
    'chosen_ensemble': chosen,
    'thresholds': {'youden': youden_thr, 'f1': f1_thr, 'f2': f2_thr},

    # NA STACKING
    'id_valid':  id_valid.astype(int),
    'rok_valid': rok_valid.astype(int),
    'y_valid':   y_valid.astype(int),
    'probs_valid_raw':   probs_valid_best,
    'probs_valid_iso':   probs_valid_cal,

    'id_test':  id_test.astype(int),
    'rok_test': rok_test.astype(int),
    'y_test':   y_test.astype(int),
    'probs_test_raw':   probs_test_best,
    'probs_test_iso':   probs_test_cal,

    # KOMPATIBILNÉ S 05_v7 stacking notebookom (alias)
    'probs_valid': probs_valid_cal,   # default = isotonic-calibrated
    'probs_test':  probs_test_cal,

    'eval_table': eval_df,
    'per_year': per_year_df,
    'thresh_table': thresh_table,
    'topk_lift': topk_lift(y_test, probs_test_cal),
    'calibration': {'raw': cs_raw, 'iso': cs_iso},
    'bootstrap_ci': bci,
    'ensemble_seeds': SEED_LIST,
    'features': {'seq': SEQ_FEATURES, 'static': STAT_FEATURES_NO_INDUSTRY,
                  'industry_cols': INDUSTRY_COLS, 'trend_cols': TREND_COLS},
}

out_pkl = os.path.join(OUTPUT_DIR, 'bilstm_v8_results.pkl')
with open(out_pkl, 'wb') as f: pickle.dump(out, f)
print(f'Saved: {out_pkl}')

# Pre stacking notebook (05_v7) — alias s názvom, ktorý hľadá
alias = os.path.join(BASE_DIR, 'bilstm_v7_results.pkl')
out_alias = dict(out)  # rovnaký dict
with open(alias, 'wb') as f: pickle.dump(out_alias, f)
print(f'Alias pre stacking: {alias}')

print('\n========== HOTOVO ==========')
print(eval_df.to_string(index=False))
print('\nNapojenie: spusti 05_v7_stacking_ensemble.ipynb — automaticky načíta tieto pickle.')

## Run summary note

This notebook prints the realised v8 metrics after execution. Historical v7 numbers are intentionally not hard-coded in the notebook, because they depend on the exact dataset version and random seed setup. Use the exported pickle files for formal model comparison.


## 19. SHAP analysis — single-seed explanatory model

For post-hoc interpretation, the notebook trains one additional seed-42 model with the best HPO hyperparameters and computes SHAP values using a gradient-based explainer. Because gradient flow through an integer embedding lookup is not directly suitable for SHAP, a mathematically equivalent one-hot industry wrapper is constructed before the SHAP calculation. The exported SHAP artefacts include global feature importance, per-time-step importance, heatmaps, beeswarm plots and selected local explanations.


In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
    import shap
print(f'SHAP version: {shap.__version__}')

In [ ]:
print('Trénujem jeden model pre SHAP analýzu (seed=42, best hparams) ...')
np.random.seed(42); tf.random.set_seed(42)
K.clear_session()

train_ds_shap, train_steps_shap = make_balanced_dataset_v8(
    X_seq_train, ind_tr, X_st_train, y_train,
    batch_size=BATCH_SIZE, pos_fraction=POS_BATCH_FRACTION, seed=42)

model_shap = build_bilstm_v8(
    SEQ_LEN, len(SEQ_FEATURES), len(STAT_FEATURES_NO_INDUSTRY), N_INDUSTRY,
    embed_dim=best['embed_dim'], lstm1=best['lstm1'], lstm2=best['lstm2'],
    dropout_rate=best['dropout'], dense_units=best['dense'],
    learning_rate=best['lr'],
    focal_gamma=best['focal_gamma'], focal_alpha=best['focal_alpha'],
    max_epochs=MAX_EPOCHS)

es_shap = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc_pr', patience=FIN_PATIENCE,
    mode='max', restore_best_weights=True, verbose=1)

t0 = time.time()
model_shap.fit(
    train_ds_shap,
    validation_data=([X_seq_valid, ind_va, X_st_valid], y_valid),
    epochs=MAX_EPOCHS, steps_per_epoch=train_steps_shap,
    callbacks=[es_shap], verbose=2)
print(f'Trénovanie SHAP modelu: {time.time()-t0:.1f}s')

# Sanity check
pv_shap = model_shap.predict([X_seq_valid, ind_va, X_st_valid], batch_size=2048, verbose=0).ravel()
pt_shap = model_shap.predict([X_seq_test,  ind_te, X_st_test],  batch_size=2048, verbose=0).ravel()
print(f'SHAP model | valid AUC={roc_auc_score(y_valid, pv_shap):.4f} '
      f'PR={average_precision_score(y_valid, pv_shap):.4f}')
print(f'SHAP model | test  AUC={roc_auc_score(y_test, pt_shap):.4f} '
      f'PR={average_precision_score(y_test, pt_shap):.4f}')

# Ulož váhy pre prípadné neskoršie načítanie / reprodukciu
weights_path = os.path.join(OUTPUT_DIR, 'bilstm_v8_shap_model.weights.h5')
model_shap.save_weights(weights_path)
print(f'Váhy SHAP modelu uložené: {weights_path}')

In [ ]:
N_BACKGROUND = 100
N_EXPLAIN = 500

rng = np.random.default_rng(42)

# === Background: stratifikovaná vzorka z train setu ===
pos_tr_idx = np.where(y_train == 1)[0]
neg_tr_idx = np.where(y_train == 0)[0]
n_pos_bg = min(50, len(pos_tr_idx))
n_neg_bg = N_BACKGROUND - n_pos_bg
bg_idx = np.concatenate([
    rng.choice(pos_tr_idx, n_pos_bg, replace=False),
    rng.choice(neg_tr_idx, n_neg_bg, replace=False),
])

bg_seq  = X_seq_train[bg_idx].astype(np.float32)
bg_ind  = ind_tr[bg_idx].reshape(-1, 1).astype(np.float32)
bg_stat = X_st_train[bg_idx].astype(np.float32)

# === Vysvetľované inštancie z TEST setu (stratifikované) ===
pos_te_idx = np.where(y_test == 1)[0]
neg_te_idx = np.where(y_test == 0)[0]
n_pos_ex = min(len(pos_te_idx), 100)
n_neg_ex = min(N_EXPLAIN - n_pos_ex, len(neg_te_idx))
ex_idx_local = np.concatenate([
    rng.choice(pos_te_idx, n_pos_ex, replace=False),
    rng.choice(neg_te_idx, n_neg_ex, replace=False),
])
rng.shuffle(ex_idx_local)

ex_seq  = X_seq_test[ex_idx_local].astype(np.float32)
ex_ind  = ind_te[ex_idx_local].reshape(-1, 1).astype(np.float32)
ex_stat = X_st_test[ex_idx_local].astype(np.float32)
ex_y    = y_test[ex_idx_local].astype(int)
ex_id   = id_test[ex_idx_local]
ex_rok  = rok_test[ex_idx_local]
ex_prob = probs_test_cal[ex_idx_local]   # iso-calibrated test probability

print(f'Background:    {len(bg_idx)} (pos={n_pos_bg}, neg={n_neg_bg})')
print(f'To explain:    {len(ex_idx_local)} (pos={n_pos_ex}, neg={n_neg_ex})')
print(f'Mean p_pred (pos vysvetľovaných): {ex_prob[ex_y==1].mean():.5f}')
print(f'Mean p_pred (neg vysvetľovaných): {ex_prob[ex_y==0].mean():.5f}')

In [ ]:
# === Wrapper s one-hot industry vstupom (kvôli gradient flow cez SHAP) ===
# GradientExplainer nedokáže pretlačiť gradient cez integer Embedding lookup.
# Riešenie: nahradíme Embedding(idx) matematicky ekvivalentným matmul(one_hot, W).
# Vstup je teraz spojitý → SHAP gradienty fungujú.

print('Building SHAP-compatible wrapper (one-hot industry input) ...')
N_IND_TOTAL = N_INDUSTRY + 1   # +1 = "missing" slot v build_bilstm_v8

# Extract embedding weights z natrénovaného modelu
emb_w = model_shap.get_layer('ind_embedding').get_weights()[0]
print(f'  embedding weights: {emb_w.shape}')

def _gl(name):
    return model_shap.get_layer(name)

# Nové vstupy (industry teraz one-hot, nie integer)
seq_in_s  = L.Input(shape=(SEQ_LEN, len(SEQ_FEATURES)), name='seq_in_s')
ind_in_s  = L.Input(shape=(N_IND_TOTAL,), name='ind_onehot_in')
stat_in_s = L.Input(shape=(len(STAT_FEATURES_NO_INDUSTRY),), name='stat_in_s')

# Sequential vetva — Masking a GaussianNoise nemajú váhy, ostatné re-use
x = L.Masking(mask_value=0.0)(seq_in_s)
x = L.GaussianNoise(0.01)(x)
x = _gl('ln_in')(x)
x = _gl('bilstm_1')(x)
x = _gl('ln_mid')(x)
x = _gl('bilstm_2')(x)
x = _gl('temporal_attn')(x)
x = _gl('ln_attn')(x)

# Industry: matmul one-hot × embedding == klasický lookup
emb_const = tf.constant(emb_w, dtype=tf.float32)
ind_emb_o = L.Lambda(lambda h: tf.matmul(h, emb_const), name='ind_matmul')(ind_in_s)

# Static vetva (re-use layers s váhami)
s = _gl('stat_dense')(stat_in_s)
s = _gl('bn_stat')(s, training=False)

# Head (re-use layers s váhami)
merged = L.Concatenate(name='concat_s')([x, ind_emb_o, s, stat_in_s])
h = _gl('head_dense')(merged)
h = _gl('bn_head')(h, training=False)
out = _gl('output')(h)

model_shap_oh = Model(inputs=[seq_in_s, ind_in_s, stat_in_s], outputs=out,
                     name='bilstm_v8_shap_wrap')

# === Sanity check: predikcie wrappera musia sedieť s originálom ===
ex_ind_oh = tf.one_hot(ex_ind.ravel().astype(int), depth=N_IND_TOTAL).numpy().astype(np.float32)
bg_ind_oh = tf.one_hot(bg_ind.ravel().astype(int), depth=N_IND_TOTAL).numpy().astype(np.float32)

p_orig = model_shap.predict([ex_seq, ex_ind, ex_stat], batch_size=512, verbose=0).ravel()
p_wrap = model_shap_oh.predict([ex_seq, ex_ind_oh, ex_stat], batch_size=512, verbose=0).ravel()
max_diff = float(np.max(np.abs(p_orig - p_wrap)))
print(f'  max abs(p_orig - p_wrap) = {max_diff:.3e}  (musí byť ≈ 0)')
assert max_diff < 1e-3, 'Wrapper nedáva rovnaké predikcie ako originál!'

# === Teraz spusti SHAP cez wrapper ===
print('\nInicializujem GradientExplainer (wrapper) ...')
explainer = shap.GradientExplainer(model_shap_oh, [bg_seq, bg_ind_oh, bg_stat])

print('Počítam SHAP values (1–3 min na CPU, <1 min na GPU) ...')
t0 = time.time()
sv = explainer.shap_values([ex_seq, ex_ind_oh, ex_stat], nsamples=200)
print(f'  hotovo za {time.time()-t0:.1f}s')

# Unpack
def _unpack_shap(sv):
    if isinstance(sv, list) and len(sv) == 3 and not isinstance(sv[0], list):
        return sv[0], sv[1], sv[2]
    if isinstance(sv, list) and len(sv) == 1 and isinstance(sv[0], list) and len(sv[0]) == 3:
        return sv[0][0], sv[0][1], sv[0][2]
    raise ValueError(f'Neočakávaný SHAP output: type={type(sv)}')

shap_seq, shap_ind_oh, shap_stat = _unpack_shap(sv)

# Odstráň trailing dim 1 (output index pre sigmoid)
if shap_seq.ndim    == 4 and shap_seq.shape[-1]    == 1: shap_seq    = shap_seq[..., 0]
if shap_stat.ndim   == 3 and shap_stat.shape[-1]   == 1: shap_stat   = shap_stat[..., 0]
if shap_ind_oh.ndim == 3 and shap_ind_oh.shape[-1] == 1: shap_ind_oh = shap_ind_oh[..., 0]

# Pre kompatibilitu s ďalšími bunkami (19.4–19.6) zhrň one-hot SHAP do (n,1):
# suma cez všetky industry sloty = celkový príspevok industry feature group
shap_ind = shap_ind_oh.sum(axis=1, keepdims=True)   # (n, 1)

print(f'\nSHAP seq:    {shap_seq.shape}    (n, T={SEQ_LEN}, n_seq={len(SEQ_FEATURES)})')
print(f'SHAP ind_oh: {shap_ind_oh.shape}  (n, N_industry+1={N_IND_TOTAL})  — full one-hot SHAP')
print(f'SHAP ind:    {shap_ind.shape}     (n, 1)  — sum cez one-hot sloty')
print(f'SHAP stat:   {shap_stat.shape}    (n, n_stat={len(STAT_FEATURES_NO_INDUSTRY)})')

In [ ]:
# === Globálna |SHAP| importance pre statické featury ===
stat_imp = np.abs(shap_stat).mean(axis=0)
stat_imp_df = pd.DataFrame({
    'feature': STAT_FEATURES_NO_INDUSTRY,
    'mean_abs_shap': stat_imp,
    'type': 'static',
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

# === Globálna pre sekvenčné — agregované cez čas ===
seq_imp_avg   = np.abs(shap_seq).mean(axis=(0, 1))             # mean over instances+time
seq_imp_max_t = np.abs(shap_seq).mean(axis=0).max(axis=0)      # max-over-time per feature
seq_imp_df = pd.DataFrame({
    'feature': SEQ_FEATURES,
    'mean_abs_shap': seq_imp_avg,
    'max_over_time_mean_abs_shap': seq_imp_max_t,
    'type': 'sequential',
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

# Industry (jediný integer → jedna hodnota |SHAP|)
ind_imp = float(np.abs(shap_ind).mean())

# Spojený ranking (statické + sekvenčné agregované + industry) pre súhrnný graf
combined = pd.concat([
    stat_imp_df[['feature', 'mean_abs_shap', 'type']],
    seq_imp_df[['feature', 'mean_abs_shap', 'type']].assign(
        feature=lambda d: d['feature'] + '  (seq)'),
    pd.DataFrame([{'feature': 'industry_idx',
                   'mean_abs_shap': ind_imp, 'type': 'embedding'}]),
], ignore_index=True).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('TOP 20 featurov podľa mean |SHAP|:')
print(combined.head(20).to_string(index=False))
print(f'\nIndustry embedding mean |SHAP|: {ind_imp:.6f}')

In [ ]:
# === Plot 1: Top-20 globálna feature importance (combined bar chart) ===
SHAP_DIR = os.path.join(OUTPUT_DIR, 'shap')
os.makedirs(SHAP_DIR, exist_ok=True)

top_n = 20
top = combined.head(top_n).iloc[::-1]   # reverse pre horizontálny bar (najsilnejší hore)
colors_map = {'static': '#1f77b4', 'sequential': '#ff7f0e', 'embedding': '#2ca02c'}
bar_colors = [colors_map[t] for t in top['type']]

fig, ax = plt.subplots(figsize=(9, max(6, top_n * 0.32)))
ax.barh(np.arange(len(top)), top['mean_abs_shap'].values,
        color=bar_colors, edgecolor='black', alpha=0.85)
ax.set_yticks(np.arange(len(top)))
ax.set_yticklabels(top['feature'].values)
ax.set_xlabel('mean |SHAP value|')
ax.set_title(f'BiLSTM v8 — Top {top_n} feature importance (SHAP)\n'
             f'background={N_BACKGROUND}, explained={len(ex_idx_local)}')
ax.grid(axis='x', alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, label=l) for l, c in colors_map.items()],
          loc='lower right')
plt.tight_layout()
out_p = os.path.join(SHAP_DIR, 'shap_global_top20.png')
plt.savefig(out_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_p}')

In [ ]:
# === Plot 2: Per-time-step |SHAP| importance ===
per_time = np.abs(shap_seq).mean(axis=(0, 2))   # priemer cez vzorky a featury
time_labels = [f'T-{SEQ_LEN-1-t}' for t in range(SEQ_LEN)]

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(time_labels, per_time, color='#ff7f0e', edgecolor='black', alpha=0.85)
for b, v in zip(bars, per_time):
    ax.text(b.get_x() + b.get_width()/2, v, f'{v:.5f}',
            ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Časový krok (T-0 = posledný rok pred predikciou)')
ax.set_ylabel('mean |SHAP value| (cez všetky featury)')
ax.set_title('BiLSTM v8 — dôležitosť časových krokov')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out_p = os.path.join(SHAP_DIR, 'shap_per_timestep.png')
plt.savefig(out_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_p}')

In [ ]:
# === Plot 3: Heatmap TOP 15 sekvenčných featurov × čas ===
top_seq = seq_imp_df.head(15)
top_seq_idx_h = [SEQ_FEATURES.index(f) for f in top_seq['feature']]

heatmap_data = np.abs(shap_seq[:, :, top_seq_idx_h]).mean(axis=0)  # (seq_len, 15)
heatmap_data = heatmap_data.T                                       # (15, seq_len)

fig, ax = plt.subplots(figsize=(7, max(6, len(top_seq) * 0.35)))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd')
ax.set_xticks(np.arange(SEQ_LEN))
ax.set_xticklabels(time_labels)
ax.set_yticks(np.arange(len(top_seq)))
ax.set_yticklabels(top_seq['feature'].values)
ax.set_xlabel('Časový krok')
ax.set_title('BiLSTM v8 — TOP 15 sekvenčných featurov × čas (mean |SHAP|)')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('mean |SHAP|')
# Anotácie hodnôt
vmax = heatmap_data.max()
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        v = heatmap_data[i, j]
        ax.text(j, i, f'{v:.4f}', ha='center', va='center', fontsize=7,
                color='black' if v < vmax * 0.6 else 'white')
plt.tight_layout()
out_p = os.path.join(SHAP_DIR, 'shap_heatmap_seq_features_time.png')
plt.savefig(out_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_p}')

In [ ]:
# === Plot 4: Beeswarm pre top-20 STATICKÝCH featurov ===
top_stat = stat_imp_df.head(20)
top_stat_idx = [STAT_FEATURES_NO_INDUSTRY.index(f) for f in top_stat['feature']]
shap_stat_top = shap_stat[:, top_stat_idx]
ex_stat_top   = ex_stat[:, top_stat_idx]

plt.figure()
shap.summary_plot(
    shap_stat_top, ex_stat_top,
    feature_names=top_stat['feature'].tolist(),
    show=False, max_display=20, plot_size=(9, 8)
)
out_p = os.path.join(SHAP_DIR, 'shap_beeswarm_static_top20.png')
plt.savefig(out_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_p}')

In [ ]:
# === Plot 5: Beeswarm pre top-20 SEKVENČNÝCH featurov (mean cez čas) ===
top20_seq_names = seq_imp_df.head(20)['feature'].tolist()
top20_seq_idx   = [SEQ_FEATURES.index(f) for f in top20_seq_names]

# Mean SHAP a hodnota cez čas (znamienkový SHAP, nie absolútny — beeswarm potrebuje smer)
shap_seq_meant = shap_seq[:, :, top20_seq_idx].mean(axis=1)   # (n, 20)
ex_seq_meant   = ex_seq[:, :, top20_seq_idx].mean(axis=1)     # (n, 20)

plt.figure()
shap.summary_plot(
    shap_seq_meant, ex_seq_meant,
    feature_names=top20_seq_names,
    show=False, max_display=20, plot_size=(9, 8)
)
out_p = os.path.join(SHAP_DIR, 'shap_beeswarm_seq_meanT_top20.png')
plt.savefig(out_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_p}')

In [ ]:
# === Plot 6: Lokálne vysvetlenia - TP / FN / FP / TN prípady ===
F1_THR_LOCAL = f1_thr   # z predošlej sekcie 15
ex_pred = (ex_prob >= F1_THR_LOCAL).astype(int)
pos_mask = ex_y == 1
neg_mask = ex_y == 0

# Vyber 4 reprezentatívne prípady
cases = {}
if pos_mask.any():
    pos_subset = np.where(pos_mask)[0]
    cases['TP_high_prob'] = int(pos_subset[np.argmax(ex_prob[pos_subset])])
    cases['FN_low_prob']  = int(pos_subset[np.argmin(ex_prob[pos_subset])])
if neg_mask.any():
    neg_subset = np.where(neg_mask)[0]
    cases['FP_high_prob'] = int(neg_subset[np.argmax(ex_prob[neg_subset])])
    cases['TN_low_prob']  = int(neg_subset[np.argmin(ex_prob[neg_subset])])

local_records = []
all_feat_names = (list(STAT_FEATURES_NO_INDUSTRY) +
                  [f + ' (seq mean t)' for f in SEQ_FEATURES] +
                  ['industry_idx'])

for label, ix in cases.items():
    seq_local_meant = shap_seq[ix].mean(axis=0)               # (n_seq,)
    contrib = np.concatenate([
        shap_stat[ix],
        seq_local_meant,
        np.array([float(shap_ind[ix].ravel()[0])]),
    ])
    df_local = pd.DataFrame({'feature': all_feat_names, 'shap': contrib})
    df_local['abs_shap'] = df_local['shap'].abs()
    df_local = df_local.sort_values('abs_shap', ascending=False).head(15).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    colors_l = ['#d62728' if v > 0 else '#1f77b4' for v in df_local['shap']]
    yp = np.arange(len(df_local))[::-1]
    ax.barh(yp, df_local['shap'], color=colors_l, edgecolor='black', alpha=0.85)
    ax.set_yticks(yp)
    ax.set_yticklabels(df_local['feature'])
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_xlabel('SHAP príspevok (smerom k riziku zlyhania)')
    ax.set_title(f'{label}  |  id={int(ex_id[ix])}  rok={int(ex_rok[ix])}  '
                 f'y={int(ex_y[ix])}  p_iso={ex_prob[ix]:.4f}')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    out_p = os.path.join(SHAP_DIR, f'shap_local_{label}.png')
    plt.savefig(out_p, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_p}')

    rec = df_local.copy()
    rec['case'] = label
    rec['firm_id'] = int(ex_id[ix])
    rec['firm_year'] = int(ex_rok[ix])
    rec['true_y'] = int(ex_y[ix])
    rec['p_pred_iso'] = float(ex_prob[ix])
    local_records.append(rec)

local_explanations_df = (pd.concat(local_records, ignore_index=True)
                         if local_records else pd.DataFrame())
print('\nLokálne vysvetlenia hotové.')

In [ ]:
# === EXPORT SHAP ÚDAJOV ===

# 1. Per-feature globálna importance (CSV)
combined.to_csv(os.path.join(SHAP_DIR, 'shap_global_importance.csv'), index=False)
stat_imp_df.to_csv(os.path.join(SHAP_DIR, 'shap_static_importance.csv'), index=False)
seq_imp_df.to_csv(os.path.join(SHAP_DIR, 'shap_seq_importance.csv'), index=False)

# 2. Per-time-step
per_time_df = pd.DataFrame({'time': time_labels, 'mean_abs_shap': per_time})
per_time_df.to_csv(os.path.join(SHAP_DIR, 'shap_per_timestep.csv'), index=False)

# 3. Per-instance SHAP — statické (wide format)
shap_stat_long = pd.DataFrame(shap_stat, columns=STAT_FEATURES_NO_INDUSTRY)
shap_stat_long.insert(0, 'firm_id', ex_id)
shap_stat_long.insert(1, 'rok', ex_rok)
shap_stat_long.insert(2, 'y_true', ex_y)
shap_stat_long.insert(3, 'p_pred_iso', ex_prob)
shap_stat_long.to_csv(os.path.join(SHAP_DIR, 'shap_per_instance_static.csv'), index=False)

# 4. Per-instance SHAP — sekvenčné (mean cez čas, wide format)
shap_seq_meant_full = shap_seq.mean(axis=1)   # (n, n_seq)
shap_seq_long = pd.DataFrame(
    shap_seq_meant_full, columns=[f + '_seqMean' for f in SEQ_FEATURES])
shap_seq_long.insert(0, 'firm_id', ex_id)
shap_seq_long.insert(1, 'rok', ex_rok)
shap_seq_long.insert(2, 'y_true', ex_y)
shap_seq_long.insert(3, 'p_pred_iso', ex_prob)
shap_seq_long.to_csv(os.path.join(SHAP_DIR, 'shap_per_instance_seq_meanT.csv'), index=False)

# 5. Per-instance SHAP — sekvenčné (long format: firm × feature × time)
seq_long_records = []
for i in range(shap_seq.shape[0]):
    for t in range(SEQ_LEN):
        for j, fname in enumerate(SEQ_FEATURES):
            seq_long_records.append({
                'firm_id': int(ex_id[i]), 'rok': int(ex_rok[i]),
                'y_true': int(ex_y[i]), 'p_pred_iso': float(ex_prob[i]),
                'feature': fname,
                'time_step': time_labels[t],
                'shap_value': float(shap_seq[i, t, j]),
                'feature_value_scaled': float(ex_seq[i, t, j]),
            })
seq_long_df = pd.DataFrame(seq_long_records)
seq_long_df.to_csv(os.path.join(SHAP_DIR, 'shap_per_instance_seq_LONG.csv'), index=False)

# 6. Industry SHAP
ind_df = pd.DataFrame({
    'firm_id': ex_id, 'rok': ex_rok, 'y_true': ex_y, 'p_pred_iso': ex_prob,
    'industry_idx': ex_ind.ravel().astype(int),
    'shap_industry': shap_ind.ravel(),
})
ind_df.to_csv(os.path.join(SHAP_DIR, 'shap_per_instance_industry.csv'), index=False)

# 7. Lokálne vysvetlenia
if not local_explanations_df.empty:
    local_explanations_df.to_csv(os.path.join(SHAP_DIR, 'shap_local_cases.csv'), index=False)

# 8. Hlavný pickle so všetkými SHAP arrays a metadátami
shap_export = {
    'meta': {
        'model':         'BiLSTM v8 (seed=42, best hparams from Optuna)',
        'shap_method':   'shap.GradientExplainer',
        'shap_version':  shap.__version__,
        'tf_version':    tf.__version__,
        'n_background':  int(len(bg_idx)),
        'n_explain':     int(len(ex_idx_local)),
        'n_pos_explain': int(n_pos_ex),
        'n_neg_explain': int(n_neg_ex),
        'nsamples':      200,
        'seq_len':       SEQ_LEN,
        'seq_features':  SEQ_FEATURES,
        'static_features': STAT_FEATURES_NO_INDUSTRY,
        'industry_cols': INDUSTRY_COLS,
        'best_hparams':  dict(best),
        'output_dir':    SHAP_DIR,
    },
    # SHAP arrays
    'shap_seq':      shap_seq,         # (n, T, n_seq)
    'shap_static':   shap_stat,        # (n, n_stat)
    'shap_industry': shap_ind,         # (n, 1)
    # Vstupy zodpovedajúce SHAP hodnotám (scaled)
    'X_seq_explained':  ex_seq,
    'X_stat_explained': ex_stat,
    'X_ind_explained':  ex_ind,
    # Identifikátory
    'firm_id':    ex_id,
    'rok':        ex_rok,
    'y_true':     ex_y,
    'p_pred_iso': ex_prob,
    # Agregované rankings
    'global_importance':       combined,
    'static_importance':       stat_imp_df,
    'seq_importance':          seq_imp_df,
    'per_timestep_importance': per_time_df,
    'industry_mean_abs_shap':  ind_imp,
    # Lokálne vysvetlenia
    'local_explanations':      local_explanations_df,
}
shap_pkl = os.path.join(SHAP_DIR, 'bilstm_v8_shap.pkl')
with open(shap_pkl, 'wb') as f:
    pickle.dump(shap_export, f)

print('\n========== SHAP EXPORT HOTOVÝ ==========')
print(f'Adresár: {SHAP_DIR}')
print('Súbory:')
for fname in [
    'shap_global_importance.csv',
    'shap_static_importance.csv',
    'shap_seq_importance.csv',
    'shap_per_timestep.csv',
    'shap_per_instance_static.csv',
    'shap_per_instance_seq_meanT.csv',
    'shap_per_instance_seq_LONG.csv',
    'shap_per_instance_industry.csv',
    'shap_local_cases.csv',
    'bilstm_v8_shap.pkl',
    'shap_global_top20.png',
    'shap_per_timestep.png',
    'shap_heatmap_seq_features_time.png',
    'shap_beeswarm_static_top20.png',
    'shap_beeswarm_seq_meanT_top20.png',
    'shap_local_TP_high_prob.png',
    'shap_local_FN_low_prob.png',
    'shap_local_FP_high_prob.png',
    'shap_local_TN_low_prob.png',
]:
    p = os.path.join(SHAP_DIR, fname)
    mark = '✓' if os.path.exists(p) else '–'
    print(f'  {mark}  {fname}')